# Chapter 1 &mdash; Convergence: Turing Machines and the Lambda Calculus

**Concept 6 of the Chapter 1 decomposition:** *Convergence of Models, and the Church-Turing Thesis*

Turing machines, the lambda calculus and Semi-Thue systems were invented independently and compute <b>exactly the same things</b>. That convergence is the evidence for the Church-Turing thesis.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Convergence-Church-Turing/Concept-Convergence-Church-Turing.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Turing proposed machines. Church proposed the **lambda calculus**. Post proposed
**Semi-Thue systems**. Three unrelated starting points, one identical class of
computable functions.

Phil Wadler's gloss: when the same powerful idea arrives from multiple perspectives,
it was **discovered, not invented**.

Here we compute the same function two ways &mdash; once with a machine-like loop, once with
pure lambdas &mdash; and check they agree.

## 2. Definitions

### Route 1: the "machine" &mdash; a step-by-step loop

State plus a rule applied repeatedly, in the spirit of a Turing machine.

In [ ]:
def machine_add(a, b):
    """Add by repeated increment -- a machine-style computation."""
    state = a
    for _ in range(b):
        state = state + 1
    return state

### Route 2: the lambda calculus &mdash; Church numerals

A number *is* a function: $n$ applies its argument $n$ times. No loops, no state,
no assignment. (Chapter 18 develops this properly.)

In [ ]:
I     = lambda c: c
ZERO  = lambda b: I
SUCC  = lambda a: lambda b: lambda c: b(a(b)(c))
ADD   = lambda a: lambda b: a(SUCC)(b)

def ChurchToNat(c):
    return c(lambda n: n + 1)(0)

def NatToChurch(n):
    out = ZERO
    for _ in range(n):
        out = SUCC(out)
    return out

## 3. Tests

Same answers from a stateful loop and from pure function application.

In [ ]:
for a, b in [(0,0), (1,0), (3,4), (7,5)]:
    lam = ChurchToNat(ADD(NatToChurch(a))(NatToChurch(b)))
    print("%d + %d :  machine=%d  lambda=%d  agree=%s"
          % (a, b, machine_add(a,b), lam, machine_add(a,b) == lam))

A broader check: the two models agree everywhere we look.

In [ ]:
agree = all(machine_add(a,b) == ChurchToNat(ADD(NatToChurch(a))(NatToChurch(b)))
            for a in range(6) for b in range(6))
print("machine and lambda agree on 0..5 x 0..5 :", agree)
assert agree
print()
print("Two formalisms, no shared machinery, same answers.")
print("That is the evidence for the Church-Turing thesis -- a THESIS, not a theorem,")
print("because 'mechanically computable' is informal and cannot be proved equal to anything.")

## 4. Exercises


1. Define `MUL` on Church numerals and check it against `a*b`.
2. `ChurchToNat` applies the numeral to "add one" and `0`. Explain in one sentence
   why that recovers the number.
3. The thesis is not a theorem. What exactly would you have to formalise to turn it
   into one &mdash; and why can't you?

In [ ]:
# Your work for the exercises above.